In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')
import os
import visual_utils_interactive as visual_utils
import utils
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

In [ ]:
data_folder = 'data/'
raw_train_df = pd.read_csv(data_folder + 'train.csv')
raw_test_df = pd.read_csv(data_folder + 'test.csv')
train_demo_df = pd.read_csv(data_folder + 'train_demographics.csv')
test_demo_df = pd.read_csv(data_folder + 'test_demographics.csv')
temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

In [ ]:
# Define sensor columns for clustering
sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'rot_w', 'rot_x', 'rot_y', 'rot_z', 
               'thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]
tof_1 = [f'tof_1_v{j}' for j in range(64)]
tof_2 = [f'tof_2_v{j}' for j in range(64)]
tof_3 = [f'tof_3_v{j}' for j in range(64)]
tof_4 = [f'tof_4_v{j}' for j in range(64)]
tof_5 = [f'tof_5_v{j}' for j in range(64)]

nan_tof_rows = raw_train_df[tof_columns].isna().any(axis=1)
nan_sequence_ids = raw_train_df.loc[nan_tof_rows, 'sequence_id'].unique()
clean_df = raw_train_df[~raw_train_df['sequence_id'].isin(nan_sequence_ids)].set_index('row_id')
clean_df['gesture_position'] = clean_df['gesture'].str.split(' - ').str[0]
clean_df['gesture_action'] = clean_df['gesture'].str.split(' - ').str[-1]

In [ ]:
explorer = visual_utils.DataExplorerApp(clean_df, train_demo_df)
explorer.show()

In [ ]:
sample_sequence_df = clean_df[clean_df['sequence_id'] == 'SEQ_065522']

dummy_extractor = utils.ImuExtractor(
    imu_sensor_list=['acc_x'],
    imu_domain='acceleration'
    )

fig, ax = plt.subplots(figsize=(10, 6), nrows = 2, ncols = 2)
ax=ax.ravel()

sample_sequence_df[dummy_extractor.imu_sensor_list].reset_index(drop=True).plot(ax=ax[0])
dummy_extractor.process_for_imu_values(sample_sequence_df).plot(ax=ax[1])
utils.ImuExtractor(imu_sensor_list=['acc_x'], imu_domain='velocity').process_for_imu_values(sample_sequence_df).plot(ax=ax[2])
utils.ImuExtractor(imu_sensor_list=['acc_x'], imu_domain='displacement').process_for_imu_values(sample_sequence_df).plot(ax=ax[3])

In [ ]:
# Check your acceleration range
print(f"Acceleration min: {sample_sequence_df['acc_x'].min():.2f}")
print(f"Acceleration max: {sample_sequence_df['acc_x'].max():.2f}")
print(f"Acceleration mean: {sample_sequence_df['acc_x'].mean():.2f}")